# NHS Waiting Times — SQL & Statistical Analysis

Walkthrough of the analysis behind the dashboard: national 4-hour A&E performance,
seasonality, the COVID shock, the RTT elective backlog, and a trust-level
correlation between emergency and elective performance.

Run `python -m src.ingest` first to build `data/nhs_waiting.db`.

In [1]:
import sys, warnings; sys.path.append('..'); warnings.simplefilter('ignore')
import pandas as pd
from src import database, analysis, stats
conn = database.connect()
print('Tables:', pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn).name.tolist())

Tables: ['ae_monthly', 'ae_national', 'rtt_monthly']


## 1. National 4-hour performance trend

In [2]:
nat = analysis.national_performance('2015-01-01')
nat.tail()

,period,att_total,att_type1,pct_within_4hrs
129,2025-10-01,2397701.0,1481222.0,74.147152
130,2025-11-01,2345860.0,1448958.0,74.202979
131,2025-12-01,2327015.0,1438921.0,73.849460
132,2026-01-01,2320266.0,1435727.0,72.460830
133,2026-02-01,2117450.0,1298005.0,74.055302


In [3]:
# Mann-Kendall trend test — is the decline statistically significant?
mk = stats.mann_kendall(nat['pct_within_4hrs'].dropna())
print(mk)

{'trend': 'decreasing', 'z': -10.13263679587567, 'p_value': 0.0, 'sen_slope': -0.19337954954366005, 's': -5269, 'n': 134}


## 2. Seasonality & COVID impact

In [4]:
analysis.seasonal_index()[['month_name', 'avg_att', 'index']]

,month_name,avg_att,index
0,Jan,1.929552e+06,96.735302
1,Feb,1.807240e+06,90.603386
2,Mar,2.052738e+06,102.911071
3,Apr,1.957449e+06,98.133891
4,May,2.082212e+06,104.388726
5,Jun,2.013199e+06,100.928850
6,Jul,2.110164e+06,105.790031
7,Aug,1.969198e+06,98.722933
8,Sep,1.973489e+06,98.938026
9,Oct,2.043888e+06,102.467369


In [5]:
# Seasonal decomposition (trend / seasonal / residual)
dec = stats.decompose(nat, 'pct_within_4hrs')
dec.tail()

,observed,trend,seasonal,residual
period,,,,
2025-10-01,74.147152,NaN,-0.468030,NaN
2025-11-01,74.202979,NaN,-1.167606,NaN
2025-12-01,73.849460,NaN,-2.761699,NaN
2026-01-01,72.460830,NaN,-2.091976,NaN
2026-02-01,74.055302,NaN,-1.325675,NaN


In [6]:
analysis.covid_comparison()

,phase,avg_4hr_performance,avg_monthly_attendances
0,1 Pre-COVID,84.7,2047083.0
1,2 During COVID,76.6,1566267.0
2,3 Post-COVID,69.9,2193829.0


## 3. Lowest-performing trusts (last 12 months)

In [7]:
analysis.worst_trusts(limit=10)

,org_name,avg_performance,avg_type1_attendances,months_reported
0,THE SHREWSBURY AND TELFORD HOSPITAL NHS TRUST,42.8,10350.0,12
1,MID CHESHIRE HOSPITALS NHS FOUNDATION TRUST,42.9,7048.0,12
2,HULL UNIVERSITY TEACHING HOSPITALS NHS TRUST,43.1,9168.0,12
3,YORK AND SCARBOROUGH TEACHING HOSPITALS NHS FO...,43.4,10743.0,12
4,ROYAL CORNWALL HOSPITALS NHS TRUST,43.9,6347.0,12
5,THE HILLINGDON HOSPITALS NHS FOUNDATION TRUST,43.9,6020.0,12
6,WIRRAL UNIVERSITY TEACHING HOSPITAL NHS FOUNDA...,45.1,7513.0,12
7,COUNTESS OF CHESTER HOSPITAL NHS FOUNDATION TRUST,46.1,5045.0,12
8,UNIVERSITY HOSPITALS PLYMOUTH NHS TRUST,46.3,9006.0,12
9,EAST AND NORTH HERTFORDSHIRE NHS TRUST,46.5,8990.0,12


## 4. RTT — worst specialties by 18-week breach

In [8]:
analysis.rtt_specialty_latest(10)[['treatment_function', 'waiting', 'pct_within_18wk', 'pct_over_52wk']]

,treatment_function,waiting,pct_within_18wk,pct_over_52wk
0,Ear Nose and Throat Service,615225,50.0,4.1
1,Oral Surgery Service,317803,50.0,4.2
2,Plastic Surgery Service,99237,53.6,3.4
3,Neurology Service,224854,54.0,2.8
4,Trauma and Orthopaedic Service,703952,54.2,3.2
5,Gynaecology Service,557932,55.4,2.9
6,General Surgery Service,368137,57.5,3.0
7,Dermatology Service,367696,57.6,2.2
8,Neurosurgical Service,54634,58.2,2.5
9,Urology Service,398260,59.3,2.9


## 5. Correlation — A&E vs RTT performance at trust level

In [9]:
corr = analysis.admission_vs_rtt()
res = stats.pearson(corr['ae_perf'], corr['rtt_perf'])
print(f"Pearson r = {res['r']:.3f}, p = {res['p_value']:.3g}, n = {res['n']}")
corr.head()

Pearson r = 0.369, p = 2.44e-05, n = 124


,org_name,admit_rate,ae_perf,rtt_perf,waiting
0,MANCHESTER UNIVERSITY NHS FOUNDATION TRUST,16.767694,53.023152,45.747392,5226781
1,SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...,30.355667,54.562582,74.252149,1614927
2,UNIVERSITY HOSPITALS DORSET NHS FOUNDATION TRUST,37.881120,64.564949,58.955281,1892777
3,ISLE OF WIGHT NHS TRUST,23.295023,54.833486,54.440506,401756
4,BARTS HEALTH NHS TRUST,14.427603,54.386187,54.537916,3287412
